In [1]:
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.path.dirname('__file__'), '..', 'database')))

from DatabaseManager import DatabaseManager
from Evaluation import Evaluation
from Doc2VecModel import Doc2VecModel

from pymongo import MongoClient

In [2]:
os.environ["MLFLOW_TRACKING_URI"] = "../../artifacts/mlruns"

In [3]:
staging_mlflow_model_readme_uri = 'models:/doc2vec_readme@staging'
staging_mlflow_model_others_uri = 'models:/doc2vec_others@staging'

production_mlflow_model_readme_uri = 'models:/doc2vec_readme@production'
production_mlflow_model_others_uri = 'models:/doc2vec_others@production'

staging_model = Doc2VecModel('staging', staging_mlflow_model_readme_uri, staging_mlflow_model_others_uri)
production_model = Doc2VecModel('production', production_mlflow_model_readme_uri, production_mlflow_model_others_uri)

In [4]:
client = MongoClient('localhost', 27017)
database_manager = DatabaseManager(client['github'])

In [5]:
evaluation = Evaluation(staging_model, production_model)

In [6]:
df_vectors = database_manager.get_df_vectors()
staging_vectors, production_vectors = database_manager.get_df_split_staging_production(df_vectors)

In [7]:
users_random_chosen_vectors, users_remaining_test_vectors = database_manager.get_users_test_vectors_splitted()

In [8]:
res = evaluation.evaluate(
    staging_vectors=staging_vectors,
    production_vectors=production_vectors,
    test_chosen_vectors=users_random_chosen_vectors,
    test_remaining_vectors=users_remaining_test_vectors,
    k=10
)

/home/choux/Documents/M2/Projet_mlops_ingestion/GitMatch/src/model/Doc2VecModel.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_vectors['similarity_score_with_readme'] = df_vectors[f'{staging}_readme_vector'].apply(lambda vector: compute_similarity_cosinus(vector, readme_vector) if len(vector) > 0 else 0)
/home/choux/Documents/M2/Projet_mlops_ingestion/GitMatch/src/model/Doc2VecModel.py:33: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_vectors['similarity_score_with_readme'] = df_vectors[f'{stagi

In [9]:
stg_scores = res[0]
prd_scores = res[1]

In [10]:
stg_scores

[0.05410106724444191,
 -0.10407411528058409,
 -0.054451590480373506,
 -0.01618922759536012]

In [11]:
prd_scores

[0.07773062003944699,
 0.011514301099389862,
 -0.07604872300313871,
 -0.007377152289968815]